# 🛡 Insurance Fraud Detection — Complete ML Pipeline

**Dataset:** insuranceFraud_Dataset.csv  
**Target:** fraud_reported (Y/N)  
**Task:** Binary Classification

## Activity 1: Collect the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import joblib, warnings
warnings.filterwarnings('ignore')
print('Libraries imported')

In [ ]:
df = pd.read_csv('insuranceFraud_Dataset.csv')
print('Shape:', df.shape)
df.head()

## Activity 2: Data Preparation

In [ ]:
df.info()

In [ ]:
df.replace('?', np.nan, inplace=True)
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum()>0])

In [ ]:
drop_cols = ['policy_number','policy_bind_date','insured_zip','incident_date','incident_location']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
df.dropna(thresh=len(df)*0.4, axis=1, inplace=True)
num_cols = df.select_dtypes(include=[np.number]).columns
obj_cols = df.select_dtypes(include='object').columns
for col in num_cols: df[col].fillna(df[col].median(), inplace=True)
for col in obj_cols: df[col].fillna(df[col].mode()[0], inplace=True)
print('After cleaning:', df.shape)
print('Nulls remaining:', df.isnull().sum().sum())

## Story 2: Visual Analysis

In [ ]:
plt.figure(figsize=(7,4))
df['fraud_reported'].value_counts().plot(kind='bar', color=['#1A7A4A','#E84545'])
plt.title('Fraud vs Non-Fraud Distribution')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
for label,color in [('Y','#E84545'),('N','#1A7A4A')]:
    df[df['fraud_reported']==label]['age'].hist(alpha=0.6,color=color,bins=20,label=label)
plt.title('Age Distribution by Fraud')
plt.legend(['Fraud','Legit'])
plt.tight_layout()
plt.show()

In [ ]:
df.boxplot(column='total_claim_amount', by='fraud_reported', figsize=(8,5))
plt.title('Claim Amount by Fraud Status')
plt.suptitle('')
plt.tight_layout()
plt.show()

## Story 2.3: Multivariate Analysis

In [ ]:
df_enc = df.copy()
df_enc['fraud_reported'] = df_enc['fraud_reported'].map({'Y':1,'N':0})
le_tmp = LabelEncoder()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le_tmp.fit_transform(df_enc[col].astype(str))
corr = df_enc.corr()['fraud_reported'].abs().sort_values(ascending=False).head(15)
plt.figure(figsize=(10,6))
corr.drop('fraud_reported').plot(kind='barh', color='#0F1F3D')
plt.title('Feature Correlations with Fraud')
plt.tight_layout()
plt.show()

## Encoding the Categorical Features

In [ ]:
df['fraud_reported'] = df['fraud_reported'].map({'Y':1,'N':0})
df.dropna(subset=['fraud_reported'], inplace=True)
df['fraud_reported'] = df['fraud_reported'].astype(int)
le_dict = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le
print('Encoding done. Shape:', df.shape)
df.head()

## Story: Scaling

In [ ]:
X = df.drop('fraud_reported', axis=1)
y = df['fraud_reported']
feature_names = X.columns.tolist()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## Story 1.1: Decision Tree Model

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'Decision Tree Accuracy: {acc_dt:.4f}')
print(classification_report(y_test, y_pred_dt, target_names=['Not Fraud','Fraud']))

## Story 1.2: Random Forest Model

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest Accuracy: {acc_rf:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Not Fraud','Fraud']))

## Activity 1.3: KNN Model

In [ ]:
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f'KNN Accuracy: {acc_knn:.4f}')
print(classification_report(y_test, y_pred_knn, target_names=['Not Fraud','Fraud']))

## Story 1.4: Logistic Regression Model

In [ ]:
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression Accuracy: {acc_lr:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Not Fraud','Fraud']))

## Story 1.5: Naive Bayes Model

In [ ]:
nb_clf = GaussianNB()
nb_clf.fit(X_train, y_train)
y_pred_nb = nb_clf.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'Naive Bayes Accuracy: {acc_nb:.4f}')
print(classification_report(y_test, y_pred_nb, target_names=['Not Fraud','Fraud']))

## Story 1.6: SVM Model

In [ ]:
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f'SVM Accuracy: {acc_svm:.4f}')
print(classification_report(y_test, y_pred_svm, target_names=['Not Fraud','Fraud']))

## Story 1.7: Testing the Model — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(15,10))
axes = axes.flatten()
models_preds = [('Decision Tree',y_pred_dt),('Random Forest',y_pred_rf),('KNN',y_pred_knn),('Logistic Regression',y_pred_lr),('Naive Bayes',y_pred_nb),('SVM',y_pred_svm)]
for idx,(name,preds) in enumerate(models_preds):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[idx], cmap='Blues', xticklabels=['Not Fraud','Fraud'], yticklabels=['Not Fraud','Fraud'])
    axes[idx].set_title(f'{name}\nAcc: {accuracy_score(y_test,preds):.2%}', fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Compare the Models

In [ ]:
results = {'Decision Tree':acc_dt,'Random Forest':acc_rf,'KNN':acc_knn,'Logistic Regression':acc_lr,'Naive Bayes':acc_nb,'SVM':acc_svm}
results_df = pd.DataFrame(list(results.items()), columns=['Model','Accuracy']).sort_values('Accuracy',ascending=False).reset_index(drop=True)
results_df['Accuracy_%'] = (results_df['Accuracy']*100).round(2)
print(results_df.to_string(index=False))
print(f'\nBest: {results_df.iloc[0]["Model"]} @ {results_df.iloc[0]["Accuracy_%"]}%')

In [ ]:
plt.figure(figsize=(10,5))
colors = ['#E84545' if i==0 else '#0F1F3D' for i in range(len(results_df))]
bars = plt.barh(results_df['Model'], results_df['Accuracy_%'], color=colors)
for bar,val in zip(bars, results_df['Accuracy_%']):
    plt.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2, f'{val}%', va='center', fontweight='bold')
plt.title('Model Accuracy Comparison', fontweight='bold')
plt.xlabel('Accuracy (%)')
plt.xlim(60,90)
plt.tight_layout()
plt.show()

## Activity 2: Save Best Model for Deployment

In [ ]:
import os
os.makedirs('models', exist_ok=True)
best_name = results_df.iloc[0]['Model']
all_models = {'Decision Tree':dt,'Random Forest':rf,'KNN':knn,'Logistic Regression':lr,'Naive Bayes':nb_clf,'SVM':svm}
best_mdl = all_models[best_name]
joblib.dump(best_mdl, 'models/best_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(le_dict, 'models/label_encoders.pkl')
joblib.dump(feature_names, 'models/feature_names.pkl')
joblib.dump(results, 'models/model_results.pkl')
joblib.dump(best_name, 'models/best_model_name.pkl')
print(f'Best model saved: {best_name}')
print('Run: python app.py  then open http://localhost:5000')